# Hard drive failure prediction

**Group One Capstone Project, 
2026, NIHUB FUT Minna**

# Modelling

**Goal.** Predict hard drive failure from daily SMART readings, and choose a model that can be used in practice: scored by drive, with its alarm level set by cost.

Decisions carried over from the EDA (`01_data_loading_and_eda`):
- **Window:** 9 consecutive days, no missing day or empty reading, for failing and healthy windows alike (the EDA used 14; 9 did as well).
- **Inputs:** SMART 5, 187, 188, 197 (core) and 4, 12, 190 (extra). `log1p` on 5, 187 and 197; 188 as a nonzero flag.
- **Dropped:** copies 198, 194 and 195; age columns 9 and 240.
- **Split:** by date, never random. Q2 2026 is not loaded here and is opened once at the very end.

In [17]:
import numpy as np
import pandas as pd

import gc, psutil
import numpy as np
import pandas as pd

def mem(): print(round(psutil.Process().memory_info().rss / 1e9, 2), "GB used by this notebook")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_curve


In [18]:

# ---------- SETTINGS: change numbers here, nowhere else ----------
SEED = 42
SEEDS = [0, 1, 2, 3, 4]            # for repeating training runs

# What one example is
WINDOW_DAYS = 9                   # days of readings the model sees
HORIZON_DAYS = 14                  # label is 1 if the drive fails within this many days AFTER the window

# Date split (proposed, not confirmed)
TRAIN_START = pd.Timestamp("2023-01-01")
TRAIN_END   = pd.Timestamp("2025-12-31")
VAL_START   = pd.Timestamp("2026-01-01")
VAL_END     = pd.Timestamp("2026-03-31")
TEST_START  = pd.Timestamp("2026-04-01")
TEST_END    = pd.Timestamp("2026-06-30")

# An example on day t looks at failures up to day t + HORIZON_DAYS.
# So the last usable example day of each period stops HORIZON_DAYS before the next period starts.
GAP = pd.Timedelta(days=HORIZON_DAYS)
TRAIN_LAST_EXAMPLE = VAL_START - GAP - pd.Timedelta(days=1)
VAL_LAST_EXAMPLE   = TEST_START - GAP - pd.Timedelta(days=1)
TEST_LAST_EXAMPLE  = TEST_END - GAP      # labels must be known inside the quarter

# Inputs
CORE = ["smart_5_raw", "smart_187_raw", "smart_188_raw", "smart_197_raw"]
CANDIDATES = ["smart_4_raw", "smart_12_raw", "smart_190_raw"]

# Cost and scoring
COST_RATIOS = [5, 10, 20, 50, 100]   # cost of a missed failure / cost of one false alarm
TOP_K = [5, 10, 25]                  # "riskiest K drives today" watch-list sizes
N_BOOTSTRAP = 1000                   # resamples for the score ranges

# ---------- sanity checks ----------
assert TRAIN_END < VAL_START <= VAL_END < TEST_START <= TEST_END
print("last train example:", TRAIN_LAST_EXAMPLE.date())
print("last validation example:", VAL_LAST_EXAMPLE.date())
print("last test example:", TEST_LAST_EXAMPLE.date())

last train example: 2025-12-17
last validation example: 2026-03-17
last test example: 2026-06-16


In [19]:
# names the loading and window cells use
EXTRA = CANDIDATES
FEATURES = CORE + CANDIDATES
LATER_QUARTERS = ["Q1_2026"]      # loaded now; Q2_2026 stays unloaded until the final test
N_NEG_TRAIN = 1_000_000           # healthy windows kept for training
DATA_ROOT = "../FailurePrediction"        # folder holding clean_train.parquet and the Dataset/ folder
DATA_DIR = f"{DATA_ROOT}/Dataset"         # folder with the quarterly parquet files

In [20]:
cols = ["serial_number", "date", "failure"] + FEATURES

train = pd.read_parquet(f"{DATA_ROOT}/clean_train.parquet", columns=cols)
train["serial_number"] = train["serial_number"].astype("category")

later = []
for q in LATER_QUARTERS:
    d = pd.read_parquet(f"{DATA_DIR}/seagate_data_{q}.parquet", columns=cols)
    d["date"] = pd.to_datetime(d["date"])
    nums = d.select_dtypes("float64").columns
    d[nums] = d[nums].astype("float32")
    later.append(d)
later = pd.concat(later, ignore_index=True)

# one shared list of drive IDs, then join
cats = pd.Index(train["serial_number"].cat.categories).union(pd.Index(later["serial_number"].unique()))
train["serial_number"] = train["serial_number"].cat.set_categories(cats)
later["serial_number"] = pd.Categorical(later["serial_number"], categories=cats)
df = pd.concat([train, later], ignore_index=True)
del train, later; _ = gc.collect()

# same cleaning rule as the EDA: each drive is cut at its first failure
df = df.sort_values(["serial_number", "date"]).reset_index(drop=True)
fail_day = df["date"].where(df["failure"] == 1)
first_fail = fail_day.groupby(df["serial_number"], observed=True).transform("min")
df = df.loc[first_fail.isna() | (df["date"] <= first_fail)].reset_index(drop=True)
del fail_day, first_fail; _ = gc.collect()

print(df.shape, "| drives:", df["serial_number"].nunique(), "| failures:", int(df["failure"].sum()))
print("dates:", df["date"].min().date(), "to", df["date"].max().date())
print(df.groupby(df["date"].dt.to_period("Q"))["failure"].sum().to_string())
mem()

(22722983, 10) | drives: 20236 | failures: 1689
dates: 2023-01-01 to 2026-03-31
date
2023Q1    120
2023Q2    157
2023Q3    170
2023Q4    126
2024Q1    144
2024Q2    122
2024Q3    126
2024Q4    110
2025Q1    104
2025Q2    138
2025Q3    132
2025Q4    111
2026Q1    129
Freq: Q-DEC
3.16 GB used by this notebook


## 1. Label design

**Question.** What is one example, and when is it labelled as failing?

**Method.**
- One example is one drive on one day (its end day). The input is that drive's last 9 days of readings, up to and including the end day, for 7 columns: SMART 5, 187, 188, 197 (core) and 4, 12, 190 (extra).
- The label is 1 if the drive fails 1 to 14 days after the end day, otherwise 0. The failure day itself never ends a window, so no input comes from the failure day or later.
- Every window needs 9 consecutive days and no empty reading, for failing and healthy windows alike.
- Healthy windows include the early days of drives that fail later, not only drives that never fail, because a data centre sees both kinds.
- All failing windows are kept. Healthy windows are sampled for training, because there are too many to keep. Validation keeps every window, so its failure share and false-alarm rate are real.
- Split by date: training windows end by 2025-12-17 and validation windows end between 2026-01-01 and 2026-03-17. The last 14 days of each period are dropped, so no label looks past the end of its period.

**Finding.** The training period gives 1,019,330 windows: 19,330 failing and 1,000,000 healthy, sampled from 20,250,835 available (each kept healthy window stands for about 20). 1,510 of the 1,560 failed drives (96.8%) have at least one failing window. Validation (Q1 2026) keeps every window: 1,389,771 windows, of which 1,136 are failing (0.082%, about 1 in 1,220), from 104 of the 129 drives that failed in the quarter. Failing windows come in runs of up to 14 per drive, so they are not independent. All leakage checks passed.

**Decision.** Keep this design. The training share (1.9%) is set by sampling; the validation share (0.082%) is the real one and is used for every score. Score ranges are computed by resampling drives, not windows. The 9-day window and 14-day horizon are settings; a 14-day window was also tried and did not do better.

In [21]:
codes = df["serial_number"].cat.codes.to_numpy()
dates = df["date"].to_numpy()
n, W, H = len(df), WINDOW_DAYS, HORIZON_DAYS

# row number within each drive (rows are contiguous and in date order)
starts = np.r_[0, np.flatnonzero(np.diff(codes) != 0) + 1]
pos = np.arange(n) - np.repeat(starts, np.diff(np.r_[starts, n]))

# days from each row to its drive's failure (a huge number if the drive never fails)
fail_idx = np.flatnonzero(df["failure"].to_numpy() == 1)
fail_date = np.full(int(codes.max()) + 1, np.datetime64("2100-01-01"), dtype=dates.dtype)
fail_date[codes[fail_idx]] = dates[fail_idx]
days_to_fail = ((fail_date[codes] - dates) / np.timedelta64(1, "D")).astype(np.int32)

# rows that can END a complete window: W consecutive days, no missing reading
X = df[FEATURES].to_numpy(np.float32)
cum = np.r_[0, np.cumsum(np.isnan(X).any(axis=1))]
first_row = np.where(pos >= W - 1, np.arange(n) - (W - 1), 0)
span = ((dates - dates[first_row]) / np.timedelta64(1, "D")).astype(np.int32)
complete = (pos >= W - 1) & (span == W - 1) & ((cum[1:] - cum[first_row]) == 0)
del cum, first_row, span; _ = gc.collect()

label = ((days_to_fail >= 1) & (days_to_fail <= H)).astype(np.int8)
usable = complete & (days_to_fail >= 1)          # the failure day itself never ends a window

def in_period(first_example, last_example):      # last_example keeps the look-ahead inside the period
    return usable & (dates >= first_example.to_datetime64()) & (dates <= last_example.to_datetime64())

rng = np.random.default_rng(SEED)
tr_mask = in_period(TRAIN_START, TRAIN_LAST_EXAMPLE)
pos_idx = np.flatnonzero(tr_mask & (label == 1))
neg_all = np.flatnonzero(tr_mask & (label == 0))
neg_idx = rng.choice(neg_all, size=min(N_NEG_TRAIN, len(neg_all)), replace=False)
train_idx = np.sort(np.r_[pos_idx, neg_idx])
val_idx = np.flatnonzero(in_period(VAL_START, VAL_LAST_EXAMPLE))

def summarize(name, idx):
    y = label[idx]
    print(f"{name:6s} windows: {len(idx):>10,} | positives: {int(y.sum()):>6,} | positive share: {y.mean() * 100:6.3f}% | drives: {len(np.unique(codes[idx])):,}")

summarize("train", train_idx); summarize("val", val_idx)
print(f"\nhealthy training windows available: {len(neg_all):,} (kept {len(neg_idx):,}; each kept one stands for {len(neg_all) / len(neg_idx):.1f})")

fi = fail_idx[dates[fail_idx] <= TRAIN_END.to_datetime64()]
print(f"failed drives in the training period: {len(fi):,} | with at least one positive window: {int(np.isin(codes[fi], codes[pos_idx]).sum()):,}")
fv = fail_idx[(dates[fail_idx] >= VAL_START.to_datetime64()) & (dates[fail_idx] <= VAL_END.to_datetime64())]
print(f"failed drives in validation: {len(fv):,} | with at least one positive window: {int(np.isin(codes[fv], codes[val_idx][label[val_idx] == 1]).sum()):,}")

assert (days_to_fail[train_idx] >= 1).all() and (days_to_fail[val_idx] >= 1).all()   # no window ends on or after the failure day
assert dates[train_idx].max() <= TRAIN_LAST_EXAMPLE.to_datetime64()                  # train labels only use train-period failures
assert dates[val_idx].max() <= VAL_LAST_EXAMPLE.to_datetime64()                      # validation labels stay inside validation
assert dates[val_idx].min() > dates[train_idx].max()                                 # validation is later than training
print("leakage checks passed")
mem()

train  windows:  1,019,330 | positives: 19,330 | positive share:  1.896% | drives: 20,211
val    windows:  1,389,771 | positives:  1,136 | positive share:  0.082% | drives: 18,617

healthy training windows available: 20,250,835 (kept 1,000,000; each kept one stands for 20.3)
failed drives in the training period: 1,560 | with at least one positive window: 1,510
failed drives in validation: 129 | with at least one positive window: 104
leakage checks passed
4.09 GB used by this notebook


## 2. Model features

**Question.** What summary of each 9-day window should the model see?

**Method.** The model gets 20 numbers per window instead of the whole sequence, taken from the EDA:
- For counters 5, 187 and 197: the end value and the change across the window (both on a `log1p` scale), plus "nonzero at the end" and "rose inside the window" flags.
- For counter 188: only the two flags (the log did not help, EDA Section 4).
- For 4 and 12: the end value (log) and whether it changed.
- For 190 (temperature): the end value and the change.

Every one of these numbers uses only the **first and last day** of the window; the days in between are not used. So the model needs two sets of readings, today's and those from 8 days earlier.

In [22]:
names = [c.replace("smart_", "").replace("_raw", "") for c in FEATURES]
J = {nm: i for i, nm in enumerate(names)}

def make_features(idx):
    first, last = X[idx - (W - 1)], X[idx]          # first and last day of each window
    f = {}
    for nm in ["5", "187", "197"]:
        a = np.log1p(np.clip(first[:, J[nm]], 0, None))
        b = np.log1p(np.clip(last[:, J[nm]], 0, None))
        f[f"{nm}_end_log"], f[f"{nm}_change_log"] = b, b - a
        f[f"{nm}_end_nz"], f[f"{nm}_rose"] = last[:, J[nm]] > 0, last[:, J[nm]] > first[:, J[nm]]
    f["188_end_nz"], f["188_rose"] = last[:, J["188"]] > 0, last[:, J["188"]] > first[:, J["188"]]
    for nm in ["4", "12"]:
        f[f"{nm}_end_log"] = np.log1p(np.clip(last[:, J[nm]], 0, None))
        f[f"{nm}_changed"] = last[:, J[nm]] != first[:, J[nm]]
    f["190_end"], f["190_change"] = last[:, J["190"]], last[:, J["190"]] - first[:, J["190"]]
    return pd.DataFrame(f).astype("float32")

F_train, F_val = make_features(train_idx), make_features(val_idx)
y_train, y_val = label[train_idx], label[val_idx]
print("features:", F_train.shape[1], "| train:", F_train.shape, "| val:", F_val.shape,
      "| NaNs:", int(F_train.isna().sum().sum() + F_val.isna().sum().sum()))

features: 20 | train: (1019330, 20) | val: (1389771, 20) | NaNs: 0


## 3. Models

**Question.** How well can simple models predict failure within 14 days, scored on validation data with the real failure share?

**Method.** Three models, fitted on the training windows and scored on every validation window:
- a rule: the number of counters (5, 187, 188, 197) that are nonzero at the end of the window;
- logistic regression on scaled features (scaler fitted on training data only);
- gradient boosting with shallow trees.

Failures are weighted up in training because they are rare. Scores: average precision (area under the precision-recall curve; a random model scores about the failure share) and the share of failing windows caught when only 1%, 0.1% or 0.01% of healthy windows raise an alarm. At 0.1% false alarms, about 1 healthy drive in 1,000 is flagged each day.

In [23]:

def recall_at_fpr(yt, s, target):
    fpr, tpr, _ = roc_curve(yt, s)
    return float(tpr[fpr <= target].max())

sc = StandardScaler().fit(F_train)                       # scaler fitted on training data only
w_pos = (y_train == 0).sum() / (y_train == 1).sum()

scores = {}
scores["rule: counters nonzero"] = F_val[["5_end_nz", "187_end_nz", "188_end_nz", "197_end_nz"]].sum(axis=1).to_numpy()
lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(sc.transform(F_train), y_train)
scores["logistic"] = lr.predict_proba(sc.transform(F_val))[:, 1]
gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.1, max_iter=200, random_state=SEED)
gb.fit(F_train, y_train, sample_weight=np.where(y_train == 1, w_pos, 1.0))
scores["grad_boost"] = gb.predict_proba(F_val)[:, 1]

rows = []
for k, s in scores.items():
    r = {"model": k, "avg_precision": average_precision_score(y_val, s)}
    for t in (0.01, 0.001, 0.0001):
        r[f"recall@{t:.2%}"] = recall_at_fpr(y_val, s, t)
    rows.append(r)
print("validation: windows", len(y_val), "| failing", int(y_val.sum()), "| failure share", round(y_val.mean() * 100, 3), "%")
print(pd.DataFrame(rows).round(3).to_string(index=False))
mem()

validation: windows 1389771 | failing 1136 | failure share 0.082 %
                 model  avg_precision  recall@1.00%  recall@0.10%  recall@0.01%
rule: counters nonzero          0.048         0.288         0.000         0.000
              logistic          0.117         0.672         0.262         0.041
            grad_boost          0.139         0.718         0.264         0.062
4.32 GB used by this notebook


**Finding.** Scored by window, at 1% false alarms the rule catches 28.8% of failing windows (none at 0.1%), logistic regression 67.2% and gradient boosting 71.8%. At 0.1%: 26.2% and 26.4%. At 0.01%: 4.1% and 6.2%. Average precision is 0.048 (rule), 0.117 (logistic) and 0.139 (gradient boosting), against about 0.001 for a random model. These are shares of failing windows; Section 4 re-scores by drive.

**Decision.** Compare the two real models by drive in Section 4, with ranges.

## 4. Drive-level scores and ranges

**Question.** Of the 129 drives that fail in Q1 2026, how many does each model catch at least once, and how much could that number change with different data?

**Method.**
- A failed drive counts as caught if any of its failing windows raises an alarm.
- The alarm level is set so that only 1%, 0.1% or 0.01% of healthy windows raise one.
- Ranges come from resampling drives 1,000 times, not windows, because one drive gives up to 14 near-identical windows.
- 25 of the 129 failed drives have no failing window (missing days or fewer than 9 days of history), so the best possible drive-level recall is 80.6%.


In [24]:
# ---------- Drive-level scores with ranges (resampling DRIVES, not windows) ----------
cv = codes[val_idx]
all_d = np.union1d(cv, codes[fv])               # every validation drive, incl. failed drives with no window
di = np.searchsorted(all_d, cv)
nd = len(all_d)
fail_drive = np.isin(all_d, codes[fv])           # the 129 drives that fail in validation
is_pos, is_neg = (y_val == 1), (y_val == 0)
pos_per = np.bincount(di[is_pos], minlength=nd)
neg_per = np.bincount(di[is_neg], minlength=nd)

rng_b = np.random.default_rng(SEED)
boot_w = [np.bincount(rng_b.integers(0, nd, nd), minlength=nd) for _ in range(N_BOOTSTRAP)]

rows = []
for name in ["logistic", "grad_boost"]:
    s = scores[name]
    for t in (0.01, 0.001, 0.0001):
        thr = np.quantile(s[is_neg], 1 - t)      # alarm when score is above this
        alarm = s > thr
        pos_alarm = np.bincount(di[is_pos & alarm], minlength=nd)
        neg_alarm = np.bincount(di[is_neg & alarm], minlength=nd)
        caught = fail_drive & (pos_alarm > 0)    # failed drive with at least one alarmed failing window

        def rng95(f):
            v = np.array([f(w) for w in boot_w]); return np.percentile(v, [2.5, 97.5])
        d_pt = caught.sum() / fail_drive.sum()
        d_lo, d_hi = rng95(lambda w: (w * caught).sum() / (w * fail_drive).sum())
        w_pt = pos_alarm.sum() / pos_per.sum()
        w_lo, w_hi = rng95(lambda w: (w * pos_alarm).sum() / (w * pos_per).sum())
        rows.append({"model": name, "false alarms": f"{t:.2%}",
                     "per 1,000 drive-days": round(t * 1000, 1),
                     "drive recall": f"{d_pt:.3f} ({d_lo:.2f}-{d_hi:.2f})",
                     "window recall": f"{w_pt:.3f} ({w_lo:.2f}-{w_hi:.2f})"})

print(f"failed validation drives: {int(fail_drive.sum())} | best possible drive recall: "
      f"{int((fail_drive & (pos_per > 0)).sum()) / int(fail_drive.sum()):.3f}")
print(pd.DataFrame(rows).to_string(index=False))
mem()

failed validation drives: 129 | best possible drive recall: 0.806
     model false alarms  per 1,000 drive-days      drive recall     window recall
  logistic        1.00%                  10.0 0.659 (0.57-0.74) 0.672 (0.58-0.75)
  logistic        0.10%                   1.0 0.380 (0.29-0.47) 0.262 (0.18-0.34)
  logistic        0.01%                   0.1 0.093 (0.04-0.14) 0.043 (0.01-0.08)
grad_boost        1.00%                  10.0 0.674 (0.59-0.75) 0.718 (0.63-0.80)
grad_boost        0.10%                   1.0 0.388 (0.31-0.47) 0.264 (0.19-0.34)
grad_boost        0.01%                   0.1 0.124 (0.07-0.18) 0.062 (0.03-0.10)
4.35 GB used by this notebook


**Finding.** Of the 129 drives that fail in Q1 2026, the best possible drive-level recall is 80.6% (104 drives have at least one failing window). At 1% false alarms (10 per 1,000 drive-days), logistic regression catches 65.9% of failed drives (range 57-74%) and gradient boosting 67.4% (59-75%). At 0.1%: 38.0% (29-47%) and 38.8% (31-47%). At 0.01%: 9.3% (4-14%) and 12.4% (7-18%). The two models' ranges overlap almost completely at every level.

**Decision.** Use logistic regression: it is no worse than gradient boosting, it is simpler, and each score can be explained reading by reading.**Finding.** Of the 129 drives that fail in Q1 2026, the best possible drive-level recall is 80.6% (104 drives have at least one failing window). At 1% false alarms (10 per 1,000 drive-days), logistic regression catches 65% of failed drives (range 56-73%) and gradient boosting 67% (58-75%). At 0.1% false alarms: 36% (29-45%) and 38% (29-46%). At 0.01%: 9% and 12%. The two models' ranges overlap almost completely at every level.

**Decision.** The CNN was compared with this baseline in 02b_cnn_experiments.ipynb and was not shown to be better, so logistic regression is the model used.

## 5. What can be caught at all

**Question.** How many of the 129 failed validation drives could any model catch from these inputs?

**Method.**
- A failed drive with no failing window (14-day gap, missing days) cannot be scored.
- A failed drive is "quiet" if all four core counters (5, 187, 188, 197) are zero at the end of every one of its failing windows.
- The remaining drives are catchable. Logistic regression is checked at the 1% false-alarm level.

In [25]:
s_log = scores["logistic"]
quiet_win = (F_val[["5_end_nz", "187_end_nz", "188_end_nz", "197_end_nz"]].sum(axis=1).to_numpy() == 0)
thr = np.quantile(s_log[is_neg], 1 - 0.01)
pa = np.bincount(di[is_pos & (s_log > thr)], minlength=nd)
caught = fail_drive & (pa > 0)
has_window = pos_per > 0
loud = np.bincount(di[is_pos & ~quiet_win], minlength=nd)
quiet_drive = fail_drive & has_window & (loud == 0)
no_window = fail_drive & ~has_window
catchable = fail_drive & has_window & ~quiet_drive
print("failed drives:", int(fail_drive.sum()))
print("  no failing window:", int(no_window.sum()), "| quiet:", int(quiet_drive.sum()), "| catchable:", int(catchable.sum()))
print(f"ceiling: {catchable.sum() / fail_drive.sum():.3f} | logistic catches {int(caught.sum())} at 1% false alarms ({caught.sum() / fail_drive.sum():.3f})")

failed drives: 129
  no failing window: 25 | quiet: 12 | catchable: 92
ceiling: 0.713 | logistic catches 85 at 1% false alarms (0.659)


**Finding.** Of the 129 failed drives, 25 have no failing window and 12 are quiet (all four core counters zero throughout), so at most 92 (71.3%) can be caught from these inputs. At 1% false alarms logistic regression catches 85 drives (65.9%), which is 85 of the 92 catchable (92%).

**Decision.** More model tuning cannot add much at this level; the remaining gap to the ceiling is small.

## 6. Alarm level by cost

**Question.** At what alarm level should the model operate if a missed failure costs R times one false alarm?

**Method.**
- Cost = false alarms + R × missed failed drives on the validation quarter. A false alarm is one healthy drive-day flagged; failed drives with no window count as missed.
- For each R in `COST_RATIOS`, 61 alarm levels are tried (never alarm, then 0.001% to 5% false alarms) and the cheapest is kept. The score threshold is chosen on validation only and frozen before the Q2 test.
- Limits: consecutive-day alarms on one drive are counted each day; choosing and reporting on the same quarter is optimistic.

In [26]:
n_val_days = len(np.unique(dates[val_idx])); n_fail = int(fail_drive.sum())
grid = np.r_[0.0, np.logspace(-5, -1.3, 60)]
curve = []
for t in grid:
    if t == 0:
        curve.append((0.0, np.inf, 0, 0)); continue
    thr = np.quantile(s_log[is_neg], 1 - t); alarm = s_log > thr
    pa = np.bincount(di[is_pos & alarm], minlength=nd)
    curve.append((float(t), float(thr), int(alarm[is_neg].sum()), int((fail_drive & (pa > 0)).sum())))

CHOSEN, rows = {}, []
for R in COST_RATIOS:
    costs = [fa + R * (n_fail - c) for (t, thr, fa, c) in curve]
    k = int(np.argmin(costs)); t, thr, fa, c = curve[k]
    CHOSEN[R] = {"false_alarm_rate": t, "threshold": thr}
    rows.append({"R": R, "false-alarm rate": f"{t:.3%}", "false alarms per day": round(fa / n_val_days, 1),
                 "failed drives caught": f"{c} of {n_fail}", "cost": int(costs[k]), "cost with no alarms": R * n_fail})
print(pd.DataFrame(rows).to_string(index=False))

  R false-alarm rate  false alarms per day failed drives caught  cost  cost with no alarms
  5           0.001%                   0.2             3 of 129   644                  645
 10           0.002%                   0.4             5 of 129  1269                 1290
 20           0.024%                   4.4            26 of 129  2393                 2580
 50           0.076%                  13.9            44 of 129  5307                 6450
100           0.157%                  28.6            59 of 129  9176                12900


**Finding.** At R = 5 an alarm is not worth it (3 of 129 caught; cost 644 against 645 with no alarms). At R = 10 it barely is (5 caught; 1,269 against 1,290). From R = 20 alarms clearly pay: R = 20 catches 26 drives with 4.4 false alarms per day (cost 2,393 against 2,580, 7% lower); R = 50 catches 44 with 13.9 per day (5,307 against 6,450, 18% lower); R = 100 catches 59 with 28.6 per day (9,176 against 12,900, 29% lower).

**Decision.** Save these alarm levels with the model. The app offers three settings: Relaxed (R = 20), Balanced (R = 50) and Cautious (R = 100).

## 6b. Today-only model

**Question.** How much is lost if the model sees only the last day's readings, not the whole window?

**Method.**
- Same training and validation windows as the 9-day model, but only the "end of window" features are kept (end values and nonzero flags: 10 of the 20), so the change over the window is not available.
- Logistic regression with the same weighting, scaler fitted on training data only.
- Scored by drive at the same false-alarm levels, with the paired comparison against the 9-day model on the same 129 failed validation drives.

In [27]:
snap_cols = [c for c in F_train.columns if c.endswith(("_end_log", "_end_nz", "_end"))]
print(len(snap_cols), "today-only features:", snap_cols)

sc_snap = StandardScaler().fit(F_train[snap_cols])
lr_snap = LogisticRegression(max_iter=2000, class_weight="balanced").fit(sc_snap.transform(F_train[snap_cols]), y_train)
s_snap = lr_snap.predict_proba(sc_snap.transform(F_val[snap_cols]))[:, 1]
s_log = scores["logistic"]                       # the model with all 20 features

def caught_at(s, t):
    thr = np.quantile(s[is_neg], 1 - t)
    pa = np.bincount(di[is_pos & (s > thr)], minlength=nd)
    return fail_drive & (pa > 0), pa

def drive_table(name, s):
    out = []
    for t in (0.01, 0.001, 0.0001):
        c, pa = caught_at(s, t)
        lo, hi = np.percentile([(w * c).sum() / (w * fail_drive).sum() for w in boot_w], [2.5, 97.5])
        out.append({"model": name, "false alarms": f"{t:.2%}",
                    "drive recall": f"{c.sum() / fail_drive.sum():.3f} ({lo:.2f}-{hi:.2f})"})
    return out

def paired(s_base, s_new, t):
    cb, _ = caught_at(s_base, t); cn, _ = caught_at(s_new, t)
    d = cn.astype(float) - cb.astype(float)
    vals = np.array([(w * d).sum() / (w * fail_drive).sum() for w in boot_w])
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return d.sum() / fail_drive.sum(), lo, hi, int((d > 0).sum()), int((d < 0).sum())

print(pd.DataFrame(drive_table("9-day model", s_log) + drive_table("today only", s_snap)).to_string(index=False))
rows_p = []
for t in (0.01, 0.001, 0.0001):
    pt, lo, hi, only_new, only_base = paired(s_log, s_snap, t)
    rows_p.append({"false alarms": f"{t:.2%}", "only today-only catches": only_new, "only 9-day catches": only_base,
                   "difference (today-only minus 9-day)": f"{pt:+.3f} ({lo:+.2f} to {hi:+.2f})"})
print(pd.DataFrame(rows_p).to_string(index=False))
print("average precision | 9-day:", round(average_precision_score(y_val, s_log), 3),
      "| today only:", round(average_precision_score(y_val, s_snap), 3))

10 today-only features: ['5_end_log', '5_end_nz', '187_end_log', '187_end_nz', '197_end_log', '197_end_nz', '188_end_nz', '4_end_log', '12_end_log', '190_end']
      model false alarms      drive recall
9-day model        1.00% 0.659 (0.57-0.74)
9-day model        0.10% 0.380 (0.29-0.47)
9-day model        0.01% 0.093 (0.04-0.14)
 today only        1.00% 0.481 (0.39-0.56)
 today only        0.10% 0.248 (0.17-0.32)
 today only        0.01% 0.101 (0.05-0.15)
false alarms  only today-only catches  only 9-day catches difference (today-only minus 9-day)
       1.00%                        0                  23             -0.178 (-0.25 to -0.11)
       0.10%                        5                  22             -0.132 (-0.21 to -0.06)
       0.01%                       10                   9             +0.008 (-0.06 to +0.08)
average precision | 9-day: 0.117 | today only: 0.113


**Finding.** Keeping only the last day's readings loses a lot. At 1% false alarms the today-only model catches 48.1% of failed drives against 65.9% for the 9-day model (difference -17.8 points, range -25 to -11; 23 drives caught only by the 9-day model, none only by today-only). At 0.1% it catches 24.8% against 38.0% (-13.2 points, range -21 to -6). At 0.01% they tie (10.1% against 9.3%). Average precision is about the same (0.113 against 0.117).

**Decision.** The 9-day model uses only the first and last day of its window, so it needs today's readings and the readings from about a week earlier, not nine days of data. The app asks for today's readings, and optionally the earlier ones. Without them, the today-only model is used, with a clear note that it is considerably less accurate.

## 8. Chance of failure

**Question.** What real-world chance of failing within 14 days does a score correspond to?

**Method.**
- The models were trained with failures weighted up, so their score is a ranking, not a probability. Weighting up makes the model behave as if failures and healthy windows were equally common.
- The real failure share among all training windows is known (about 1 in 1,000), so the score is converted to a real chance by multiplying the odds by that share.
- The conversion is checked on validation: for each range of predicted chance, the share of windows that really failed. Failing windows come in runs for the same drive, so the check is approximate.

In [28]:
pi_real = len(pos_idx) / (len(pos_idx) + len(neg_all))      # real share of failing windows in the training period
print(f"real failing share, training: {pi_real:.5f} (about 1 in {1 / pi_real:,.0f}) | validation: {y_val.mean():.5f}")

def to_chance(score, pi):
    s = np.clip(score, 1e-9, 1 - 1e-9)
    odds = s / (1 - s) * (pi / (1 - pi))
    return odds / (1 + odds)

bins = [0, 0.0003, 0.0006, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 1.0]
for name, s in [("9-day model", s_log), ("today-only model", s_snap)]:
    p = to_chance(s, pi_real)
    tbl = (pd.DataFrame({"predicted": p, "actual": y_val, "range": pd.cut(p, bins=bins, include_lowest=True)})
             .groupby("range", observed=True).agg(windows=("actual", "size"), predicted=("predicted", "mean"), actual=("actual", "mean")))
    tbl["predicted"], tbl["actual"] = tbl["predicted"].round(4), tbl["actual"].round(4)
    print("\n" + name); print(tbl.to_string())

real failing share, training: 0.00095 (about 1 in 1,049) | validation: 0.00082

9-day model
                  windows  predicted  actual
range                                       
(-0.001, 0.0003]  1130267     0.0002  0.0001
(0.0003, 0.0006]   136317     0.0004  0.0001
(0.0006, 0.001]     14625     0.0008  0.0002
(0.001, 0.002]      55613     0.0013  0.0005
(0.002, 0.005]      15627     0.0033  0.0039
(0.005, 0.01]       19514     0.0071  0.0035
(0.01, 0.02]         5073     0.0138  0.0126
(0.02, 0.05]         5716     0.0325  0.0114
(0.05, 0.1]          1456     0.0705  0.0405
(0.1, 1.0]           5563     0.4135  0.1088

today-only model
                  windows  predicted  actual
range                                       
(-0.001, 0.0003]  1152302     0.0002  0.0001
(0.0003, 0.0006]    86176     0.0005  0.0002
(0.0006, 0.001]     35112     0.0007  0.0002
(0.001, 0.002]      43225     0.0016  0.0010
(0.002, 0.005]      28415     0.0031  0.0027
(0.005, 0.01]        5213     0.007

**Finding.** The conversion is accurate for low and middle scores (9-day model: predicted 0.33% vs actual 0.39%; predicted 1.4% vs actual 1.3%). For the highest scores it is too confident: the 9-day top group was predicted at 41% but 11% actually failed within 14 days; for the today-only model 24% was predicted and 6% actually failed. The likely reason is that the models look more certain on new drives than on the drives they trained on (not proven).

**Decision.** The app will not show the converted number. It will show the failure rate measured on the validation quarter for drives with a similar score ("about 1 in N"), using a rising lookup curve fitted on validation. The Q2 2026 test, if run, will check it.

In [29]:
from sklearn.isotonic import IsotonicRegression

iso = {}
for name, s in [("nine_day", s_log), ("today", s_snap)]:
    p = to_chance(s, pi_real)
    iso[name] = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(p, y_val)
    print("\n" + name)
    for q in [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 0.6, 0.9]:
        c = float(iso[name].predict([q])[0])
        print(f"converted {q:<7} -> measured {c:.4f}", f"(about 1 in {1 / c:,.0f})" if c > 0 else "(no failures seen)")


nine_day
converted 0.0003  -> measured 0.0002 (about 1 in 5,938)
converted 0.001   -> measured 0.0002 (about 1 in 5,938)
converted 0.003   -> measured 0.0036 (about 1 in 278)
converted 0.01    -> measured 0.0091 (about 1 in 110)
converted 0.03    -> measured 0.0144 (about 1 in 70)
converted 0.1     -> measured 0.0399 (about 1 in 25)
converted 0.3     -> measured 0.0945 (about 1 in 11)
converted 0.6     -> measured 0.1288 (about 1 in 8)
converted 0.9     -> measured 0.1925 (about 1 in 5)

today
converted 0.0003  -> measured 0.0002 (about 1 in 5,054)
converted 0.001   -> measured 0.0004 (about 1 in 2,393)
converted 0.003   -> measured 0.0027 (about 1 in 372)
converted 0.01    -> measured 0.0073 (about 1 in 137)
converted 0.03    -> measured 0.0073 (about 1 in 137)
converted 0.1     -> measured 0.0244 (about 1 in 41)
converted 0.3     -> measured 0.0540 (about 1 in 19)
converted 0.6     -> measured 0.7857 (about 1 in 1)
converted 0.9     -> measured 1.0000 (about 1 in 1)


## 9. Alarm levels for the today-only model

**Question.** At what alarm level should the today-only model operate for each cost ratio R?

**Method.** The same as Section 6: cost = false alarms + R × missed failed drives on the validation quarter, the cheapest of 61 alarm levels kept for each R, threshold frozen before any later test.

In [30]:
curve_s = []
for t in grid:
    if t == 0:
        curve_s.append((0.0, np.inf, 0, 0)); continue
    thr = np.quantile(s_snap[is_neg], 1 - t); alarm = s_snap > thr
    pa = np.bincount(di[is_pos & alarm], minlength=nd)
    curve_s.append((float(t), float(thr), int(alarm[is_neg].sum()), int((fail_drive & (pa > 0)).sum())))

CHOSEN_snap, rows_snap = {}, []
for R in COST_RATIOS:
    costs = [fa + R * (n_fail - c) for (t, thr, fa, c) in curve_s]
    k = int(np.argmin(costs)); t, thr, fa, c = curve_s[k]
    CHOSEN_snap[R] = {"false_alarm_rate": t, "threshold": thr}
    rows_snap.append({"R": R, "false-alarm rate": f"{t:.3%}", "false alarms per day": round(fa / n_val_days, 1),
                      "failed drives caught": f"{c} of {n_fail}", "cost": int(costs[k]), "cost with no alarms": R * n_fail})
print(pd.DataFrame(rows_snap).to_string(index=False))

  R false-alarm rate  false alarms per day failed drives caught  cost  cost with no alarms
  5           0.001%                   0.2             3 of 129   644                  645
 10           0.001%                   0.2             3 of 129  1274                 1290
 20           0.010%                   1.8            13 of 129  2460                 2580
 50           0.016%                   2.8            16 of 129  5866                 6450
100           0.088%                  16.1            31 of 129 11021                12900


**Finding.** The today-only model costs more than the 9-day model at every R from 20 up (R = 20: 2,460 against 2,393; R = 50: 5,866 against 5,307; R = 100: 11,021 against 9,176). At R = 100 it catches 31 of 129 failed drives with 16.1 false alarms per day; the 9-day model catches 59 with 28.6.

**Decision.** Save these alarm levels too. The app uses them only when no earlier readings are given.

## 10. Warning levels (cautious setting)

**Question.** How many healthy drives get a warning, and how many failing drives are caught, if the one-reading card flags the top 0.1%, 0.5%, 1%, 2%, 5%, 10% or 20% of healthy drive-days?

**Method.**
- For each share of healthy drive-days flagged, the score threshold is set on the validation quarter.
- A failed drive counts as caught if any of its failing windows is flagged; ranges come from resampling drives.
- Also shown: how many healthy drives are flagged per day, and what share of all flagged drive-days failed within 14 days.
- Both models are shown. The one-reading card uses the one-reading model.

In [33]:
rows_c = []
for name, s_ in [("9-day model", s_log), ("one-reading model", s_snap)]:
    for t in (0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20):
        thr_ = np.quantile(s_[is_neg], 1 - t)
        flagged = s_ > thr_
        c_, _ = caught_at(s_, t)
        lo, hi = np.percentile([(w * c_).sum() / (w * fail_drive).sum() for w in boot_w], [2.5, 97.5])
        rate_ = float(y_val[flagged].mean())
        rows_c.append({"model": name, "healthy drive-days flagged": f"{t:.1%}",
                       "healthy drives flagged per day": round(flagged[is_neg].sum() / n_val_days, 1),
                       "failing drives caught": f"{int(c_.sum())} of {n_fail} ({c_.sum() / n_fail:.0%}, range {lo:.0%}-{hi:.0%})",
                       "of flagged drive-days, 1 in": round(1 / rate_) if rate_ > 0 else None})
print(pd.DataFrame(rows_c).to_string(index=False))

            model healthy drive-days flagged  healthy drives flagged per day          failing drives caught  of flagged drive-days, 1 in
      9-day model                       0.1%                            18.3 49 of 129 (38%, range 29%-47%)                            6
      9-day model                       0.5%                            91.4 78 of 129 (60%, range 52%-69%)                           11
      9-day model                       1.0%                           182.7 85 of 129 (66%, range 57%-74%)                           19
      9-day model                       2.0%                           365.2 87 of 129 (67%, range 59%-75%)                           34
      9-day model                       5.0%                           913.6 91 of 129 (71%, range 63%-78%)                           75
      9-day model                      10.0%                          1827.0 92 of 129 (71%, range 63%-78%)                          146
      9-day model                      20

In [34]:
LEVELS = {"red": 0.01, "yellow": 0.05}      # share of healthy drive-days flagged
bands, rows_b = {}, []
for name, s_ in [("9-day model", s_log), ("one-reading model", s_snap)]:
    thr_red = np.quantile(s_[is_neg], 1 - LEVELS["red"])
    thr_yel = np.quantile(s_[is_neg], 1 - LEVELS["yellow"])
    bands[name] = {"red": float(thr_red), "yellow": float(thr_yel)}
    band = np.where(s_ > thr_red, "red", np.where(s_ > thr_yel, "yellow", "green"))
    for b in ("red", "yellow", "green"):
        m = band == b
        rate_ = float(y_val[m].mean())
        rows_b.append({"model": name, "level": b, "healthy drive-days": f"{m[is_neg].mean():.1%}",
                       "healthy drives per day": round(m[is_neg].sum() / n_val_days, 1),
                       "drive-days that failed within 14 days: 1 in": round(1 / rate_) if rate_ > 0 else None})
    c_red, _ = caught_at(s_, LEVELS["red"])
    c_yel, _ = caught_at(s_, LEVELS["yellow"])
    print(f"{name}: failing drives red at some point: {int(c_red.sum())} of {n_fail}; "
          f"red or yellow: {int(c_yel.sum())} of {n_fail}")
print(pd.DataFrame(rows_b).to_string(index=False))

9-day model: failing drives red at some point: 85 of 129; red or yellow: 91 of 129
one-reading model: failing drives red at some point: 62 of 129; red or yellow: 87 of 129
            model  level healthy drive-days  healthy drives per day  drive-days that failed within 14 days: 1 in
      9-day model    red               1.0%                   182.7                                           19
      9-day model yellow               4.0%                   730.9                                          328
      9-day model  green              95.0%                 17357.9                                         6500
one-reading model    red               1.0%                   182.7                                           24
one-reading model yellow               4.0%                   730.8                                          198
one-reading model  green              95.0%                 17358.0                                         5498


**Finding.** With the red level at the worst 1% and the yellow level at the next 4% of healthy drive-days, the one-reading model puts 48% of the 129 failing drives in red at some point (62 drives) and 67% in red or yellow (87 drives). Drive-days in the red band failed within 14 days about 1 time in 24, yellow about 1 in 200 and green about 1 in 5,500 (a normal drive-day: about 1 in 1,200). For the 9-day model the figures are 66% and 71% (85 and 91 drives), and 1 in 19, 1 in 330 and 1 in 6,500. About 183 healthy drives a day are red and 731 yellow, out of about 18,600.

**Decision.** The cautious cards use these levels. The "out of every N drives" sentence uses the measured rate of each band. These cut-offs were chosen and measured on the same quarter, so real results may be somewhat lower; the Q2 test will check this.

In [35]:
import joblib
for key, name, path, s_ in [("nine_day", "9-day model", "model/nine_day_model.joblib", s_log),
                            ("today", "one-reading model", "model/today_model.joblib", s_snap)]:
    thr_red, thr_yel = bands[name]["red"], bands[name]["yellow"]
    band = np.where(s_ > thr_red, "red", np.where(s_ > thr_yel, "yellow", "green"))
    lv = {"red": {"threshold": thr_red}, "yellow": {"threshold": thr_yel}, "green": {"threshold": None}}
    for b in ("red", "yellow", "green"):
        lv[b]["rate"] = float(y_val[band == b].mean())
    art = joblib.load(path)
    art["levels"] = lv
    joblib.dump(art, path)
    print(name, "saved; 1 in", {b: round(1 / lv[b]["rate"]) for b in lv})

9-day model saved; 1 in {'red': 19, 'yellow': 328, 'green': 6500}
one-reading model saved; 1 in {'red': 24, 'yellow': 198, 'green': 5498}


## 10. Files for the app

**Question.** What does the app need so that it can score a drive without retraining?

**Method.** Save both models (9-day and today-only), their alarm levels by cost ratio, the lookup curves that turn a score into "about 1 in N", the cost tables, the validation numbers, and example drives built from 9-day windows. The lookup curve is capped: scores above the top 0.5% of validation windows share one rate, because the very top has too few drives to trust (without the cap, the curve claimed certainty).

In [31]:
import joblib, json, os
from sklearn.isotonic import IsotonicRegression

os.makedirs("model", exist_ok=True)

# ---- lookup curves: score -> measured failure rate, capped at the top 0.5% ----
lookup = {}
for name, s in [("nine_day", s_log), ("today", s_snap)]:
    p = to_chance(s, pi_real)
    p_cap = float(np.quantile(p, 0.995))
    iso_m = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip").fit(np.minimum(p, p_cap), y_val)
    lookup[name] = {"iso": iso_m, "p_cap": p_cap}
    top = float(iso_m.predict([p_cap])[0])
    print(f"{name}: cap at converted {p_cap:.3f}; at or above it, about 1 in {1 / top:,.0f} failed within {H} days")
    for q in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 0.9]:
        c = float(iso_m.predict([min(q, p_cap)])[0])
        print(f"   converted {q:<6} -> measured {c:.4f}", f"(about 1 in {1 / c:,.0f})" if c > 0 else "")

# ---- models ----
joblib.dump({"model": lr, "scaler": sc, "features": list(F_train.columns),
             "window_days": int(W), "horizon_days": int(H), "alarm_levels": CHOSEN,
             "pi_real": float(pi_real), "lookup": lookup["nine_day"]},
            "model/nine_day_model.joblib")
joblib.dump({"model": lr_snap, "scaler": sc_snap, "features": list(snap_cols),
             "window_days": 1, "horizon_days": int(H), "alarm_levels": CHOSEN_snap,
             "pi_real": float(pi_real), "lookup": lookup["today"]},
            "model/today_model.joblib")
pd.DataFrame(rows).to_csv("model/cost_table.csv", index=False)
pd.DataFrame(rows_snap).to_csv("model/cost_table_today.csv", index=False)

# ---- validation summary ----
summary = {"validation_quarter": "2026-Q1", "failed_drives": n_fail, "no_window": int(no_window.sum()),
           "quiet": int(quiet_drive.sum()), "catchable": int(catchable.sum()),
           "drive_recall": {"nine_day": {}, "today": {}}}
for key, s_ in [("nine_day", s_log), ("today", s_snap)]:
    for t in (0.01, 0.001, 0.0001):
        c_, _ = caught_at(s_, t)
        lo, hi = np.percentile([(w * c_).sum() / (w * fail_drive).sum() for w in boot_w], [2.5, 97.5])
        summary["drive_recall"][key][f"{t:.2%}"] = {"recall": float(c_.sum() / n_fail), "low": float(lo), "high": float(hi)}
json.dump(summary, open("model/validation_summary.json", "w"), indent=2)

# ---- demo drives (raw readings, Backblaze-style columns), 9-day windows ----
rng_demo = np.random.default_rng(SEED)
vcodes, vlab = codes[val_idx], label[val_idx]
demo_rows, demo_key = [], []

def add_window(end_row, name, failed):
    for k in range(W):
        r = end_row - (W - 1) + k
        demo_rows.append({"serial_number": name, "date": pd.Timestamp(dates[r]).date(),
                          **{c: float(X[r, j]) for j, c in enumerate(FEATURES)}})
    i = np.searchsorted(val_idx, end_row)
    demo_key.append({"serial_number": name, "fails_within_14_days": failed,
                     "offline_score_9day": float(s_log[i]), "offline_score_today": float(s_snap[i])})

n_f = 0
for code in rng_demo.permutation(np.unique(codes[fv])):
    rows_c = val_idx[(vcodes == code) & (vlab == 1)]
    if len(rows_c) == 0:
        continue
    add_window(rows_c[np.argmin(np.abs(days_to_fail[rows_c] - 7))], f"demo_fail_{n_f + 1:02d}", True)
    n_f += 1
    if n_f == 6:
        break
healthy_pool = val_idx[(vlab == 0) & (days_to_fail[val_idx] > 10_000)]
for i, end_row in enumerate(rng_demo.choice(healthy_pool, 14, replace=False)):
    add_window(end_row, f"demo_ok_{i + 1:02d}", False)

pd.DataFrame(demo_rows).to_csv("model/demo_drives.csv", index=False)
pd.DataFrame(demo_key).to_csv("model/demo_key.csv", index=False)
print(sorted(os.listdir("model")))
print(pd.DataFrame(demo_key).round(3).to_string(index=False))

nine_day: cap at converted 0.051; at or above it, about 1 in 10 failed within 14 days
   converted 0.001  -> measured 0.0002 (about 1 in 5,938)
   converted 0.003  -> measured 0.0036 (about 1 in 278)
   converted 0.01   -> measured 0.0091 (about 1 in 110)
   converted 0.03   -> measured 0.0144 (about 1 in 70)
   converted 0.1    -> measured 0.0954 (about 1 in 10)
   converted 0.3    -> measured 0.0954 (about 1 in 10)
   converted 0.9    -> measured 0.0954 (about 1 in 10)
today: cap at converted 0.136; at or above it, about 1 in 17 failed within 14 days
   converted 0.001  -> measured 0.0004 (about 1 in 2,393)
   converted 0.003  -> measured 0.0027 (about 1 in 372)
   converted 0.01   -> measured 0.0073 (about 1 in 137)
   converted 0.03   -> measured 0.0073 (about 1 in 137)
   converted 0.1    -> measured 0.0541 (about 1 in 18)
   converted 0.3    -> measured 0.0585 (about 1 in 17)
   converted 0.9    -> measured 0.0585 (about 1 in 17)
['cost_table.csv', 'cost_table_today.csv', 'demo_d

**Finding.** After capping, the lookup curves are sensible. For the 9-day model, scores in the top 0.5% of validation windows correspond to about 1 failure in 10 within 14 days; for the today-only model, about 1 in 17. Lower scores range down to about 1 in 5,000 (9-day) and 1 in 2,400 (today-only). The 6 example failing drives all score above 0.89 under the 9-day model; the 14 example healthy drives score 0.17 to 0.58.

**Decision.** The app reports "about 1 in N" from these curves and the alarm from the cost-based alarm levels (Sections 6 and 9). The raw score is never shown as a probability. The example drives were picked to illustrate, not at random; the app says so.

## 11. Reading ranges for the app

**Question.** What reading values did the models see in training, so the app can warn about inputs outside them and offer sensible example values?

**Method.** Percentiles of the last-day reading for healthy windows (the training sample) and failing windows.

**Finding.** SMART 188 holds huge raw numbers (up to 1.3 trillion), which is why it is used only as zero or non-zero. Temperature is mostly 25 to 46 °C. The largest values seen in training: SMART 5: 65,368; 187: 65,535 (looks like a 16-bit limit); 197: 335,008; 4: 3,870; 12: 3,868; 190: 62 °C. Half of failing windows have 32 or more reallocated sectors and 10 or more uncorrectable errors; the worst tenth have 5 at 6,856, 187 at 158 and 197 at 96.

**Decision.** The app warns when an input is above these maxima, and its example drives use these typical values.

In [32]:
qs = [50, 90, 99, 99.9, 100]
def show(rows_idx, title):
    sub = X[rows_idx]
    out = pd.DataFrame({c.replace("smart_", "").replace("_raw", ""): np.nanpercentile(sub[:, j], qs)
                        for j, c in enumerate(FEATURES)}, index=[f"{q}%" for q in qs])
    print(title); print(out.round(1).to_string()); print()
show(neg_idx, "Healthy windows (reading on the last day)")
show(pos_idx, "Failing windows (reading on the last day)")

Healthy windows (reading on the last day)
             5      187           188      197       4      12   190
50%        0.0      0.0  0.000000e+00      0.0    11.0    10.0  31.0
90%        0.0      0.0  0.000000e+00      0.0    21.0    20.0  40.0
99%     1112.0     13.0  2.577020e+10      0.0    73.0    55.0  46.0
99.9%  22632.0    185.0  2.534071e+11     56.0  2644.0   252.0  51.0
100%   63792.0  65535.0  1.301395e+12  11200.0  3870.0  3868.0  62.0

Failing windows (reading on the last day)
             5      187           188       197       4      12   190
50%       32.0     10.0  0.000000e+00       0.0    11.0    11.0  32.0
90%     6856.0    158.0  2.147516e+10      96.0    21.0    20.0  41.0
99%    37214.7    581.7  2.534081e+11    2584.0    79.7    73.0  47.0
99.9%  61671.7  65535.0  8.890718e+11   26224.0  1370.1  1348.8  51.0
100%   65368.0  65535.0  1.357263e+12  335008.0  1883.0  1802.0  57.0



## 12. Conclusion (validation, Q1 2026)

- Model: logistic regression on 20 summary features of a 9-day window (only its first and last day); the label is "fails within the next 14 days". It performs about as well as gradient boosting, and it is simpler and explainable.
- At 1% false alarms it catches 65.9% of the 129 failed validation drives (range 57 to 74%); at 0.1%, 38.0% (29 to 47%); at 0.01%, 9.3% (4 to 14%).
- At most 92 of the 129 (71.3%) can be caught from these inputs: 25 have no usable window and 12 look healthy on all four core counters.
- With only today's readings the model is clearly weaker: 48.1% at 1% false alarms, 17.8 points lower (range 11 to 25).
- The alarm level depends on the cost ratio R. From R = 20 alarms pay off (R = 100: 28.6 false alarms per day, 59 of 129 failed drives caught).
- The app reports "about 1 in N" from the failure rate measured on validation, not the raw score. Even for the highest scores, only about 1 in 10 drives (1 in 17 for today-only) failed within 14 days.
- Limits: Q2 2026 (April to June) was not used for training, choosing features, or setting the warning levels. A small random sample of Q2 drive-days was used for a live demonstration. The full Q2 evaluation has not been run.